# Course 05 lab — requirements engineering for coding agents

Northstar Mutual ticket AI-2176 asks an AI assistant to identify missing underwriting information, draft a broker follow-up, and perhaps send simple cases automatically. We will turn that request into evidence-backed requirements and a bounded runtime decision. The notebook is deterministic, offline, and never sends a message.

## Objectives and safety boundary

You will inspect ambiguity, validate a structured requirement package, compare a lexical baseline with evidence-aware rules, test output correspondence, inject stale approval failures, and examine task traceability. The model or agent proposes; trusted code validates and authorizes. A simulated `PROCEED` is not a delivery receipt.

In [ ]:
import importlib.util, sys
from dataclasses import replace
from datetime import UTC, datetime
from pathlib import Path

spec = importlib.util.spec_from_file_location('course05_lab', Path('lab.py'))
lab = importlib.util.module_from_spec(spec)
assert spec.loader
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
package = lab.load_package()
package['specification']

## 1 — A polished ticket is still ambiguous

The baseline flags visible vague words and an unbounded side effect. It cannot discover every missing stakeholder decision, so treat its findings as review candidates rather than truth.

In [ ]:
ticket = (lab.SCENARIO_ROOT / 'ticket' / 'AI-2176.md').read_text(encoding='utf-8')
ticket_findings = lab.baseline_ticket_findings(ticket)
assert 'UNIVERSAL_QUANTIFIER_REVIEW' in {item.code for item in ticket_findings}
[(item.code, item.message) for item in ticket_findings]

## 2 — Validate structure, provenance, and requirement quality

The reference requirements avoid the ticket's vague vocabulary, preserve owners and sources, separate priority from normative strength, and name an evidence method. Passing these checks does not prove completeness.

In [ ]:
quality = lab.requirement_quality_findings(package)
assert not [item for item in quality if item.severity is lab.Severity.STOP]
[(item['id'], item['normative_strength'], item['priority']) for item in package['requirements']]

## 3 — Block the capability, not the whole change

`OQ-017` is a policy decision about external delivery. Analysis and drafting do not depend on that authority, so they remain ready while send is deferred and disabled. Requirement lifecycle, release applicability, and capability readiness are independent: the send controls remain approved durable requirements even though they are not active in this release.

In [ ]:
readiness = {item.capability: item for item in lab.capability_readiness(package)}
assert readiness['analyze_missing_items'].ready
assert readiness['draft_message'].ready
assert not readiness['send_message'].ready
assert readiness['send_message'].deferred_requirement_ids == ('REL-FU-002', 'SEC-FU-003')
{key: (value.ready, value.release_status, value.blocking_question_ids) for key, value in readiness.items()}

## 4 — Validate the domain plan before generation

The authoritative rule set determines obligations. A language model may propose typed `RequirementGap` records, but deterministic code rejects unsupported requirements and stale observations. Rule evidence is separate from a revision-bound observation of absence, invalidity, unverifiability, or conflict.

In [ ]:
gaps, draft, approval, context = lab.demo_objects(package)
observed = {gap.observation.id for gap in gaps}
assert not lab.validate_requirement_gaps(gaps, current_requirement_ids={'REQ-FU-001'}, trusted_requirement_evidence_ids={'SRC-UW-REQ-12'}, current_submission_revision='7', observed_evidence_ids=observed)
unsupported = gaps + (replace(gaps[0], id='GAP-003', requirement_id='REQ-NOT-CURRENT'),)
[item.code for item in lab.validate_requirement_gaps(unsupported, current_requirement_ids={'REQ-FU-001'}, trusted_requirement_evidence_ids={'SRC-UW-REQ-12'}, current_submission_revision='7', observed_evidence_ids=observed)]

## 5 — Measure draft correspondence

Precision reveals unsupported additions; recall reveals omissions. Reporting both with denominators avoids hiding broker burden behind one aggregate score.

In [ ]:
perfect = lab.correspondence_metrics((gap.id for gap in gaps), draft.gap_ids)
faulty = lab.correspondence_metrics((gap.id for gap in gaps), ('GAP-001', 'GAP-999'))
assert perfect['precision'].value == perfect['recall'].value == 1.0
{'perfect': perfect, 'injected_failure': faulty}

## 6 — Bind approval to exact, current content

The happy path satisfies the simulated preconditions. Then we mutate the body after approval: the digest changes and delivery must fail closed.

In [ ]:
now = datetime(2026, 9, 20, 18, 0, tzinfo=UTC)
allowed = lab.authorize_send(draft, approval, context, now=now)
changed = replace(draft, body=draft.body + ' Also provide bank statements.')
blocked = lab.authorize_send(changed, approval, context, now=now)
assert allowed.outcome is lab.Outcome.PROCEED
assert blocked.outcome is lab.Outcome.BLOCK
assert 'DRAFT_CHANGED_AFTER_APPROVAL' in blocked.reason_codes
{'happy_path': allowed, 'changed_after_approval': blocked}

## 7 — Make retries and terminal behavior explicit

A transient transport failure can use a bounded retry with the same logical operation ID. Schema, safety, evidence, and quality failures do not become permission to try again blindly.

In [ ]:
first_transport = lab.failure_decision(package, 'transport', attempts=0)
exhausted_transport = lab.failure_decision(package, 'transport', attempts=2)
safety = lab.failure_decision(package, 'safety', attempts=0)
assert first_transport.reason_codes == ('BOUNDED_RETRY_ALLOWED',)
assert safety.outcome is lab.Outcome.ESCALATE
[first_transport, exhausted_transport, safety]

## 8 — Evaluate the review aid, not the language model

The labelled synthetic cases compare wording-only lint with rules that also inspect source support, ownership, and observability. This measures the teaching detector on this fixture—not model quality or production safety.

In [ ]:
evaluation = lab.evaluate_cases()
assert evaluation['evidence_aware']['recall'] >= evaluation['keyword_baseline']['recall']
evaluation

## 9 — Trace work and detect stale context

The reference tasks cover every approved requirement through many-to-many links. A pinned old specification revision raises a review finding; impact analysis decides whether work must be replanned.

In [ ]:
assert not lab.traceability_findings(package)
[item.code for item in lab.spec_context_findings(package, '90-old')]

## Production upgrade and exercises

| Course fixture | Production requirement |
| --- | --- |
| synthetic source metadata | authenticated source registry, freshness, and owner workflow |
| in-memory decision | transactional state and atomic single-use approval consumption |
| simulated send gate | narrow delivery adapter, reconciliation, provider receipt, and audit |
| eight labelled cases | versioned representative data, slices, owner-approved thresholds, and drift monitoring |
| local timestamps | trusted clock and explicit expiry semantics |

Exercises:

1. Add `conflicting` and `unverified` gap fixtures and prove they cannot be collapsed into `missing`.
2. Add an expired-approval test; revise a requirement and inspect the effective-context digest; introduce an orphan task; define evidence for increasing delivery autonomy.
3. Decompose this over-specified statement: ‘The system SHALL use LangChain StructuredOutputParser with GPT-5.6 behind FastAPI and persist results in Redis.’ Keep the outcome as a requirement and classify the parser, API framework, persistence store, and model as separately owned design, architecture, persistence, and technology decisions.
4. Engineer the under-specified statement ‘Follow up with brokers appropriately.’ Discover the actor, trigger, input, authoritative source, behavior, failure states, authorization boundary, and conformance evidence without granting send authority.

## Summary

Good requirements reduce decision entropy without pretending uncertainty has disappeared. They preserve sources, owners, boundaries, states, failure behavior, evidence plans, and authority. Coding agents can move quickly inside that envelope; trusted application code still owns consequential decisions and effects.